# 🤖 Demo: Roteamento Dinâmico com LLM
**Encontro 2 - Grupo de NLP**

Esta demo mostra um **Orquestrador LLM** que roteia tarefas para sub-agentes especializados usando dois padrões:

| Padrão | Como funciona |
|--------|---------------|
| `agent_as_tool` | Orquestrador chama o sub-agente como ferramenta e processa a resposta |
| `transfer_to_agent` | Orquestrador transfere o controle completo para o sub-agente |

**Sub-agentes:**
- 🔍 `summarizer_agent` — resume textos
- 💬 `sentiment_agent` — analisa sentimento
- 🌐 `translator_agent` — traduz textos

## 1. Instalação

In [ ]:
!pip install groq -q

## 2. Configuração

In [ ]:
from groq import Groq
from openai import OpenAI
import json

# ── Orquestrador: Llama 3.3 70B via Groq ─────────────────────────
GROQ_API_KEY      = "sua_chave_aqui"
ORCHESTRATOR_MODEL = "llama-3.3-70b-versatile"

# ── Sub-agentes: GAIA 4B PT-BR via HuggingFace ───────────────────
HF_TOKEN   = "sua_chave_aqui"
AGENT_MODEL = "CEIA-UFG/Gemma-3-Gaia-PT-BR-4b-it:featherless-ai"

orchestrator_client = Groq(api_key=GROQ_API_KEY)

agent_client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=HF_TOKEN,
)

print("✅ Configuração pronta!")
print(f"   Orquestrador : {ORCHESTRATOR_MODEL} via Groq (70B)")
print(f"   Sub-agentes  : GAIA PT-BR via HuggingFace (4B)")


## 3. Sub-Agentes Especializados

In [ ]:
def call_agent(system_prompt: str, user_message: str, label: str) -> str:
    """Helper genérico para chamar um sub-agente GAIA."""
    print(f"    {label} ativado")
    response = agent_client.chat.completions.create(
        model=AGENT_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ],
        max_tokens=400
    )
    return response.choices[0].message.content


def summarizer_agent(text: str) -> str:
    return call_agent(
        system_prompt=(
            "Você é um especialista em sumarização de textos em português. "
            "Gere um resumo conciso e fiel ao conteúdo original."
        ),
        user_message=f"Resuma o seguinte texto:\n\n{text}",
        label="📋 [summarizer_agent / GAIA]"
    )


def sentiment_agent(text: str) -> str:
    return call_agent(
        system_prompt=(
            "Você é um especialista em análise de sentimento em português. "
            "Identifique: sentimento predominante (positivo/negativo/neutro), "
            "intensidade emocional e emoções específicas. "
            "Justifique com trechos do texto."
        ),
        user_message=f"Analise o sentimento deste texto:\n\n{text}",
        label="💬 [sentiment_agent / GAIA]"
    )


def translator_agent(text: str, target_language: str = "inglês") -> str:
    print(f"    🌐 [translator_agent / GAIA] ativado → traduzindo para {target_language}")
    return call_agent(
        system_prompt=(
            "Você é um tradutor profissional. "
            "Mantenha tom, estilo e nuances do original. "
            "Responda APENAS com a tradução, sem explicações."
        ),
        user_message=f"Traduza para {target_language}:\n\n{text}",
        label=f"🌐 [translator_agent / GAIA] → {target_language}"
    )


AGENT_MAP = {
    "summarizer_agent": summarizer_agent,
    "sentiment_agent":  sentiment_agent,
    "translator_agent": translator_agent,
}

print("✅ Sub-agentes prontos usando GAIA (UFG/CEIA, 4B params, PT-BR)")


## 4. Orquestrador — Padrão `agent_as_tool`

O orquestrador **não usa if/else** para decidir a rota.
Ele expõe os sub-agentes como `tools` e deixa o LLM escolher qual chamar.
Após receber o resultado, o orquestrador formula a **resposta final**.

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "summarizer_agent",
            "description": "Resume um texto. Use quando o usuário pedir resumo, síntese ou visão geral.",
            "parameters": {
                "type": "object",
                "properties": {
                    "text": {"type": "string", "description": "Texto a ser resumido"}
                },
                "required": ["text"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "sentiment_agent",
            "description": "Analisa sentimento e emoções de um texto. Use quando o usuário quiser saber o tom emocional ou polaridade.",
            "parameters": {
                "type": "object",
                "properties": {
                    "text": {"type": "string", "description": "Texto para análise de sentimento"}
                },
                "required": ["text"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "translator_agent",
            "description": "Traduz um texto para outro idioma. Use quando o usuário pedir tradução.",
            "parameters": {
                "type": "object",
                "properties": {
                    "text": {"type": "string", "description": "Texto a ser traduzido"},
                    "target_language": {"type": "string", "description": "Idioma destino (ex: inglês, espanhol, francês)"}
                },
                "required": ["text", "target_language"]
            }
        }
    }
]


def orchestrator(user_request: str):
    """
    Orquestrador com roteamento dinâmico (agent_as_tool).
    Fluxo completo:
      1. Orquestrador decide qual sub-agente chamar (roteamento via LLM)
      2. Sub-agente executa a tarefa (GAIA 4B)
      3. Orquestrador recebe o resultado e formula resposta final
    """
    print("=" * 60)
    print(f"📥 Requisição: {user_request[:80].strip()}...")
    print("=" * 60)
    print("🧠 [orchestrator] analisando intenção...")

    messages = [
        {
            "role": "system",
            "content": (
                "Você é um orquestrador de agentes NLP. "
                "Analise a requisição e use a ferramenta mais adequada. "
                "Sempre use uma das ferramentas disponíveis."
            )
        },
        {"role": "user", "content": user_request}
    ]

    # Passo 1: Orquestrador decide qual tool usar (roteamento dinâmico via LLM)
    response = orchestrator_client.chat.completions.create(
        model=ORCHESTRATOR_MODEL,
        messages=messages,
        tools=TOOLS,
        tool_choice="auto",
        max_tokens=500
    )

    message = response.choices[0].message

    if not message.tool_calls:
        # Entrada não requer nenhum sub-agente (ex: saudação)
        print("💬 [orchestrator] respondeu diretamente (sem sub-agente necessário).")
        print("-" * 60)
        print(message.content)
        print("=" * 60)
        return message.content

    tool_call  = message.tool_calls[0]
    agent_name = tool_call.function.name
    agent_args = json.loads(tool_call.function.arguments)

    print(f"🔀 [orchestrator] roteou para → {agent_name}")

    # Passo 2: Sub-agente executa a tarefa (GAIA)
    agent_result = AGENT_MAP[agent_name](**agent_args)
    print(f"\n📨 [orchestrator] recebeu resultado do {agent_name}, processando...")

    # Passo 3: Orquestrador recebe o resultado e formula resposta final
    messages.append({"role": "assistant", "tool_calls": message.tool_calls})
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": agent_result
    })
    messages.append({
        "role": "system",
        "content": (
            "O sub-agente retornou o resultado acima. "
            "Apresente esse resultado ao usuário de forma clara e direta, "
            "adicionando uma frase introdutória sobre qual agente processou a tarefa. "
            "NÃO ignore o resultado. NÃO invente informações novas."
        )
    })

    final = orchestrator_client.chat.completions.create(
        model=ORCHESTRATOR_MODEL,
        messages=messages,
        max_tokens=600
    )

    result = final.choices[0].message.content
    print("\n📤 Resposta final do orquestrador:")
    print("-" * 60)
    print(result)
    print("=" * 60)
    return result


print("✅ Orquestrador pronto com pós-processamento (agent_as_tool completo)")


## 5. Demo — 3 Rotas Diferentes

In [ ]:
# ── CASO 1: Rota → summarizer_agent ──────────────────────────────
orchestrator("""
Me dê um resumo deste trecho:

A inteligência artificial generativa tem transformado profundamente a forma como interagimos
com sistemas computacionais. Modelos de linguagem de grande escala, como GPT e Claude,
são capazes de gerar texto coerente, responder perguntas complexas e até escrever código.
Esses avanços têm aplicações em saúde, educação, jurídico e entretenimento,
mas também levantam questões éticas sobre viés, privacidade e o futuro do trabalho humano.
""")

In [ ]:
# ── CASO 2: Rota → sentiment_agent ───────────────────────────────
orchestrator("""
Qual o sentimento deste comentário de cliente?

Fiquei muito desapontado com o produto. A entrega atrasou uma semana,
a embalagem chegou amassada e o atendimento foi completamente indiferente
à minha reclamação. Não recomendo e não voltarei a comprar.
""")

In [ ]:
# ── CASO 3: Rota → translator_agent ──────────────────────────────
orchestrator("""
Traduza para inglês:

Redes neurais artificiais são sistemas computacionais inspirados no cérebro humano,
compostos por camadas de neurônios artificiais que aprendem padrões a partir de dados.
""")

## 6. Comparativo — Padrão `transfer_to_agent`

Aqui o orquestrador **cede o controle** completamente.
O sub-agente responde direto ao usuário — sem passar pelo orquestrador novamente.

**Diferença chave:** no `agent_as_tool` o orquestrador pós-processa a resposta.
No `transfer_to_agent` ele apenas decide quem vai falar.

In [ ]:
def orchestrator_transfer(user_request: str):
    """
    Single-step Router — simplificação do padrão transfer_to_agent.
    O orquestrador decide qual agente responde, mas não pós-processa.
    Diferença real vs agent_as_tool: sem segunda passagem pelo orquestrador.

    Nota: em produção, o transfer_to_agent real (ex: OpenAI Swarm) transfere
    o histórico completo da conversa — o sub-agente assume múltiplos turnos.
    Aqui implementamos single-turn por simplicidade.
    """
    print("=" * 60)
    print(f"📥 Requisição: {user_request[:80].strip()}...")
    print("=" * 60)
    print("🧠 [orchestrator] decidindo para qual agente transferir...")

    routing = orchestrator_client.chat.completions.create(
        model=ORCHESTRATOR_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Você é um roteador de agentes. "
                    "Responda APENAS com o nome do agente mais adequado, nada mais. "
                    "Opções: summarizer_agent | sentiment_agent | translator_agent"
                )
            },
            {"role": "user", "content": user_request}
        ],
        max_tokens=20
    )

    raw = routing.choices[0].message.content.strip().lower()
    chosen = next((name for name in AGENT_MAP if name in raw), None)

    if not chosen:
        print(f"⚠️  Parsing falhou: '{raw}'. Usando sentiment_agent como fallback.")
        chosen = "sentiment_agent"

    print(f"🔀 [transfer] → controle cedido para: {chosen}")
    print("   (orquestrador NÃO pós-processa — sub-agente responde direto)")

    # Reutiliza a função do AGENT_MAP — mesmos prompts do agent_as_tool
    result = AGENT_MAP[chosen](user_request)

    print(f"\n📤 Resposta direta do {chosen}:")
    print("-" * 60)
    print(result)
    print("=" * 60)
    return result


# Testando
orchestrator_transfer("""
Analise o sentimento:

Que experiência incrível! O produto superou todas as minhas expectativas,
chegou antes do prazo e a qualidade é excepcional. Com certeza voltarei a comprar!
""")


## 7. Trade-offs

| Critério | `agent_as_tool` | `transfer_to_agent` |
|---|---|---|
| **Controle** | Orquestrador mantém controle total | Controle cedido ao sub-agente |
| **Custo (tokens)** | 3 chamadas LLM | 2 chamadas LLM |
| **Latência** | Maior (vai e volta) | Menor (resposta direta) |
| **Pós-processamento** | ✅ Orquestrador pode reformular | ❌ Resposta do sub-agente é final |
| **Debugabilidade** | Mais fácil — estado intermediário visível | Mais difícil — caixa-preta |
| **Uso ideal** | Síntese de múltiplos agentes | Agente auto-contido e especializado |

### Trade-offs gerais de sistemas multi-agente

- **Custo**: cada agente = chamada extra → cresce com hierarquia
- **Previsibilidade**: roteamento dinâmico pode tomar caminhos inesperados
- **Debugabilidade**: mais agentes = mais difícil rastrear o fluxo
- **Latência**: agentes em cadeia somam os tempos de cada chamada
- **Escalabilidade**: fácil adicionar novos agentes sem mudar o orquestrador

## 8. Análise de Custos

Comparação entre as arquiteturas usando preços reais dos providers (junho/2026).

In [ ]:
# ── Preços reais (junho/2026) ─────────────────────────────────────
# Groq — Llama 3.3 70B
LLAMA_INPUT_PER_M  = 0.59   # USD por 1M tokens de entrada
LLAMA_OUTPUT_PER_M = 0.79   # USD por 1M tokens de saída

# HuggingFace Inference Router — GAIA 4B (via Featherless)
# Modelos pequenos (~4B) custam ~$0.10/M input, ~$0.10/M output
GAIA_INPUT_PER_M   = 0.10
GAIA_OUTPUT_PER_M  = 0.10

# ── Estimativa de tokens por requisição (média da demo) ──────────
# Uma requisição típica da demo tem ~200 tokens de entrada e ~150 de saída
TOKENS_IN  = 200   # tokens de entrada por chamada
TOKENS_OUT = 150   # tokens de saída por chamada

def custo_chamada(input_per_m, output_per_m, tokens_in=TOKENS_IN, tokens_out=TOKENS_OUT):
    return (tokens_in / 1_000_000 * input_per_m) + (tokens_out / 1_000_000 * output_per_m)

# ── Custo por arquitetura ─────────────────────────────────────────

# 1. LLM único (sem agentes) — 1 chamada ao Llama 70B
custo_llm_unico = custo_chamada(LLAMA_INPUT_PER_M, LLAMA_OUTPUT_PER_M)

# 2. transfer_to_agent — 2 chamadas: Llama (roteamento) + GAIA (execução)
custo_transfer = (
    custo_chamada(LLAMA_INPUT_PER_M, LLAMA_OUTPUT_PER_M) +   # Llama roteia
    custo_chamada(GAIA_INPUT_PER_M,  GAIA_OUTPUT_PER_M)      # GAIA responde
)

# 3. agent_as_tool — 3 chamadas: Llama (roteamento) + GAIA (execução) + Llama (síntese)
custo_agent_as_tool = (
    custo_chamada(LLAMA_INPUT_PER_M, LLAMA_OUTPUT_PER_M) +   # Llama roteia
    custo_chamada(GAIA_INPUT_PER_M,  GAIA_OUTPUT_PER_M)  +   # GAIA executa
    custo_chamada(LLAMA_INPUT_PER_M, LLAMA_OUTPUT_PER_M)     # Llama pós-processa
)

# 4. if/else determinístico — 1 chamada só ao GAIA (sem roteamento LLM)
custo_ifelse = custo_chamada(GAIA_INPUT_PER_M, GAIA_OUTPUT_PER_M)

# ── Projeção para escala ──────────────────────────────────────────
requisicoes_dia   = 10_000
requisicoes_mes   = requisicoes_dia * 30

arquiteturas = {
    "if/else + GAIA 4B":          custo_ifelse,
    "LLM único (Llama 70B)":      custo_llm_unico,
    "transfer_to_agent":          custo_transfer,
    "agent_as_tool (completo)":   custo_agent_as_tool,
}

# ── Tabela de resultados ──────────────────────────────────────────
print("=" * 72)
print(f"{'ANÁLISE DE CUSTOS — ARQUITETURAS MULTI-AGENTE':^72}")
print("=" * 72)
print(f"Premissas: {TOKENS_IN} tokens entrada / {TOKENS_OUT} tokens saída por chamada")
print(f"Llama 3.3 70B (Groq): ${LLAMA_INPUT_PER_M}/M input, ${LLAMA_OUTPUT_PER_M}/M output")
print(f"GAIA 4B (HuggingFace): ${GAIA_INPUT_PER_M}/M input, ${GAIA_OUTPUT_PER_M}/M output")
print("-" * 72)
print(f"{'Arquitetura':<28} {'Chamadas':>8} {'$/req':>10} {'$/dia (10k)':>13} {'$/mês':>12}")
print("-" * 72)

n_chamadas = {
    "if/else + GAIA 4B":        1,
    "LLM único (Llama 70B)":    1,
    "transfer_to_agent":        2,
    "agent_as_tool (completo)": 3,
}

for nome, custo in arquiteturas.items():
    custo_dia = custo * requisicoes_dia
    custo_mes = custo * requisicoes_mes
    print(f"{nome:<28} {n_chamadas[nome]:>8}x  ${custo*1000:>7.4f}/1k  ${custo_dia:>10.2f}  ${custo_mes:>10.2f}")

print("-" * 72)

# ── Multiplicadores de custo ──────────────────────────────────────
base = arquiteturas["if/else + GAIA 4B"]
print("\nMultiplicador de custo vs if/else + GAIA 4B:")
for nome, custo in arquiteturas.items():
    mult = custo / base
    print(f"  {nome:<30} {mult:.1f}x")

print("=" * 72)
print("\n💡 Conclusão:")
print("   • agent_as_tool custa ~8x mais que um if/else simples")
print("   • transfer_to_agent é ~4x mais caro que if/else")  
print("   • Usar GAIA 4B nos sub-agentes reduz o custo vs usar Llama 70B em tudo")
print("   • O custo extra justifica-se pela flexibilidade do roteamento dinâmico")
print("     e pela capacidade de lidar com intenções ambíguas")
